# Étape 2 - Inventaire technique des sources retenues

Profile `store_sales`, `item` et `store` : taille, lignes, colonnes, types inférés vs attendus, valeurs nulles, unicité des clés candidates et premières anomalies.

Les fichiers sources sont seulement lus. Résultat : `outputs/inventaire_sources.json`.

In [1]:
import os

# Racine du projet : on remonte depuis le dossier du notebook
RACINE = os.getcwd()
# Racine du depot = dossier qui contient config/parametres.json
while not os.path.exists(os.path.join(RACINE, "config", "parametres.json")):
    if os.path.dirname(RACINE) == RACINE:
        raise RuntimeError("config/parametres.json introuvable : lancer le notebook depuis le depot")
    RACINE = os.path.dirname(RACINE)

# Sans le prefixe file://, Spark irait chercher les fichiers dans HDFS
# (fs.defaultFS = hdfs://localhost:9000 dans la configuration Hadoop)
LOCAL = "file://"
import json

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SOURCE = os.path.expanduser("~/DEV/tpcxbb_source/retail_project/data")
SCHEMA_SPEC = os.path.join(RACINE, "01_UC01_besoin_exploration", "preuves", "tpcxbb_schema.json")
SORTIE = os.path.join(RACINE, "outputs", "inventaire_sources.json")

# Source -> cle candidate
SOURCES = {
    "store_sales": ["ss_ticket_number", "ss_item_sk"],
    "item": ["i_item_sk"],
    "store": ["s_store_sk"],
}

# Noms et types attendus (specification TPCx-BB relevee au Jour 1)
with open(SCHEMA_SPEC) as f:
    SPEC = json.load(f)

In [2]:
spark = (
    SparkSession.builder
    .appName("TP3-Inventaire-Sources")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 14:28:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Lecture et profilage de chaque source

In [3]:
def lire(table):
    """Lit un fichier .dat (delimite par |, sans en-tete) avec les noms de la specification."""
    chemin = os.path.join(SOURCE, table, f"{table}.dat")
    noms = [c[0] for c in SPEC[table]]
    brut = spark.read.csv(LOCAL + chemin, sep="|", header=False, inferSchema=True)
    return chemin, len(brut.columns), brut.select(brut.columns[: len(noms)]).toDF(*noms)


def profiler(table, cle):
    chemin, nb_colonnes_fichier, df = lire(table)
    df = df.cache()
    noms = df.columns

    print(f"===== {table} =====")
    df.printSchema()
    df.show(3, truncate=30)

    nb_lignes = df.count()

    # Valeurs nulles (et chaines vides) par colonne
    nulls = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
        for c in noms
    ]).first().asDict()

    # Unicite de la cle candidate
    nb_cles_distinctes = df.select(cle).distinct().count()

    infos = {
        "chemin": chemin,
        "taille_octets": os.path.getsize(chemin),
        "nb_lignes": nb_lignes,
        "nb_colonnes": len(noms),
        "nb_colonnes_lues_dans_fichier": nb_colonnes_fichier,
        "types_inferes": dict(df.dtypes),
        "types_attendus": {c[0]: c[1] for c in SPEC[table]},
        "valeurs_nulles": {c: int(v) for c, v in nulls.items() if v},
        "cle_candidate": cle,
        "cles_distinctes": nb_cles_distinctes,
        "doublons_cle": nb_lignes - nb_cles_distinctes,
        "lignes_entierement_dupliquees": nb_lignes - df.dropDuplicates().count(),
        "controles": controles(table, df),
    }
    df.unpersist()
    return infos

## Contrôles propres à chaque source

In [4]:
def controles(table, df):
    ctrl = {}
    if table == "store_sales":
        ctrl["quantite_nulle_ou_negative"] = df.filter(F.col("ss_quantity") <= 0).count()
        ctrl["prix_vente_negatif"] = df.filter(F.col("ss_sales_price") < 0).count()
        ctrl["prix_vente_superieur_prix_catalogue"] = df.filter(
            F.col("ss_sales_price") > F.col("ss_list_price")).count()
        ctrl["montant_incoherent_qte_x_prix"] = df.filter(
            F.abs(F.col("ss_ext_sales_price") - F.col("ss_quantity") * F.col("ss_sales_price")) > 0.01
        ).count()
        ctrl["lignes_marge_negative"] = df.filter(F.col("ss_net_profit") < 0).count()
        ctrl["nb_tickets"] = df.select("ss_ticket_number").distinct().count()
        ctrl["nb_magasins"] = df.select("ss_store_sk").distinct().count()
        ctrl["nb_produits_vendus"] = df.select("ss_item_sk").distinct().count()
        bornes = df.agg(F.min("ss_sold_date_sk"), F.max("ss_sold_date_sk")).first()
        ctrl["date_sk_min_max"] = [bornes[0], bornes[1]]
    elif table == "item":
        ctrl["i_item_id_distincts"] = df.select("i_item_id").distinct().count()
        ctrl["nb_categories"] = df.select("i_category").distinct().count()
        ctrl["categories"] = sorted(
            r[0] for r in df.select("i_category").distinct().collect() if r[0] is not None)
        ctrl["prix_courant_negatif_ou_nul"] = df.filter(F.col("i_current_price") <= 0).count()
        ctrl["cout_superieur_prix"] = df.filter(
            F.col("i_wholesale_cost") > F.col("i_current_price")).count()
    elif table == "store":
        ctrl["s_store_id_distincts"] = df.select("s_store_id").distinct().count()
        ctrl["magasins_fermes"] = df.filter(F.col("s_closed_date_sk").isNotNull()).count()
        ctrl["etats"] = sorted(
            r[0] for r in df.select("s_state").distinct().collect() if r[0] is not None)
    return ctrl

In [5]:
resultats = {table: profiler(table, cle) for table, cle in SOURCES.items()}

===== store_sales =====
root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: double (nullable = true)
 |-- ss_list_price: double (nullable = true)
 |-- ss_sales_price: double (nullable = true)
 |-- ss_ext_discount_amt: double (nullable = true)
 |-- ss_ext_sales_price: double (nullable = true)
 |-- ss_ext_wholesale_cost: double (nullable = true)
 |-- ss_ext_list_price: double (nullable = true)
 |-- ss_ext_tax: double (nullable = true)
 |-- ss_coupon_amt: double (nullable = true)
 |-- ss_net_paid: double (nullable = true)


+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

===== item =====
root
 |-- i_item_sk: integer (nullable = true)
 |-- i_item_id: string (nullable = true)
 |-- i_rec_start_date: date (nullable = true)
 |-- i_rec_end_date: string (nullable = true)
 |-- i_item_desc: string (nullable = true)
 |-- i_current_price: double (nullable = true)
 |-- i_wholesale_cost: double (nullable = true)
 |-- i_brand_id: integer (nullable = true)
 |-- i_brand: string (nullable = true)
 |-- i_class_id: integer (nullable = true)
 |-- i_class: string (nullable = true)
 |-- i_category_id: integer (nullable = true)
 |-- i_category: string (nullable = true)
 |-- i_manufact_id: integer (nullable = true)
 |-- i_manufact: string (nullable = true)
 |-- i_size: string (nullable = true)
 |-- i_formulation: string (nullable = true)
 |-- i_color: string (nullable = true)
 |-- i_units: string (nullable = true)
 |-- i_container: string (nullable = true)
 |-- i_manager_id: integer (nullable = true)
 |-- i_product_name: string (nullable = true)



+---------+----------------+----------------+--------------+------------------------------+---------------+----------------+----------+------------------------------+----------+---------------------+-------------+-----------------+-------------+------------------------------+------+--------------------+-------+-------+-----------+------------+------------------------------+
|i_item_sk|       i_item_id|i_rec_start_date|i_rec_end_date|                   i_item_desc|i_current_price|i_wholesale_cost|i_brand_id|                       i_brand|i_class_id|              i_class|i_category_id|       i_category|i_manufact_id|                    i_manufact|i_size|       i_formulation|i_color|i_units|i_container|i_manager_id|                i_product_name|
+---------+----------------+----------------+--------------+------------------------------+---------------+----------------+----------+------------------------------+----------+---------------------+-------------+-----------------+-------------+-

## Relations : les jointures sont-elles vérifiables ?

In [6]:
ventes = lire("store_sales")[2]
items = lire("item")[2]
magasins = lire("store")[2]

resultats["relations"] = {
    "store_sales.ss_item_sk sans correspondance dans item":
        ventes.join(items, ventes.ss_item_sk == items.i_item_sk, "left_anti").count(),
    "store_sales.ss_store_sk sans correspondance dans store":
        ventes.join(magasins, ventes.ss_store_sk == magasins.s_store_sk, "left_anti").count(),
}
resultats["relations"]

{'store_sales.ss_item_sk sans correspondance dans item': 0,
 'store_sales.ss_store_sk sans correspondance dans store': 0}

## Synthèse et écriture du résultat

In [7]:
os.makedirs(os.path.dirname(SORTIE), exist_ok=True)
with open(SORTIE, "w") as f:
    json.dump(resultats, f, indent=1, ensure_ascii=False)

print(json.dumps({t: {k: v for k, v in r.items() if k not in ("types_attendus", "types_inferes")}
                  for t, r in resultats.items()}, indent=1, ensure_ascii=False))
print("Resultats ecrits dans", SORTIE)

{
 "store_sales": {
  "chemin": "/home/lolorigolo/DEV/tpcxbb_source/retail_project/data/store_sales/store_sales.dat",
  "taille_octets": 92764700,
  "nb_lignes": 667579,
  "nb_colonnes": 23,
  "nb_colonnes_lues_dans_fichier": 23,
  "valeurs_nulles": {
   "ss_cdemo_sk": 238,
   "ss_hdemo_sk": 295,
   "ss_addr_sk": 448,
   "ss_promo_sk": 158
  },
  "cle_candidate": [
   "ss_ticket_number",
   "ss_item_sk"
  ],
  "cles_distinctes": 663443,
  "doublons_cle": 4136,
  "lignes_entierement_dupliquees": 0,
  "controles": {
   "quantite_nulle_ou_negative": 0,
   "prix_vente_negatif": 0,
   "prix_vente_superieur_prix_catalogue": 0,
   "montant_incoherent_qte_x_prix": 0,
   "lignes_marge_negative": 495971,
   "nb_tickets": 89100,
   "nb_magasins": 12,
   "nb_produits_vendus": 17820,
   "date_sk_min_max": [
    36890,
    38697
   ]
  }
 },
 "item": {
  "chemin": "/home/lolorigolo/DEV/tpcxbb_source/retail_project/data/item/item.dat",
  "taille_octets": 6799286,
  "nb_lignes": 17820,
  "nb_colonnes"

In [8]:
spark.stop()